# NYC Yellow Taxi Data Understanding (Q1 2025)

This notebook examines the final analysis-ready NYC Yellow Taxi dataset before model building.

## 1. Import Libraries and Load the Dataset

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import pearsonr, spearmanr

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

DATA_FILE = Path("nyc_yellow_taxi_master_analysis_ready_2025_q1.csv")
if not DATA_FILE.exists():
    DATA_FILE = Path("/mnt/data/nyc_yellow_taxi_master_analysis_ready_2025_q1.csv")

df = pd.read_csv(DATA_FILE)
df["pickup_hour"] = pd.to_datetime(df["pickup_hour"])

display(df.head())

FileNotFoundError: [Errno 2] No such file or directory: '\\mnt\\data\\nyc_yellow_taxi_master_analysis_ready_2025_q1.csv'

**Interpretation:** The analysis-ready master dataset is loaded with the hourly taxi demand, temporal variables, weather variables, spatial identifiers, contextual features, and Census quality fields required for Data Understanding.

## 2. Dataset Observations

In [ ]:
zone_counts = df.groupby("LocationID").size()

observation_summary = pd.DataFrame({
    "Measure": [
        "Rows",
        "Columns",
        "Taxi zones",
        "Unique hourly timestamps",
        "Study start",
        "Study end",
        "Rows per zone - minimum",
        "Rows per zone - maximum",
        "Duplicate zone-hour records",
        "Missing cells"
    ],
    "Result": [
        len(df),
        df.shape[1],
        df["LocationID"].nunique(),
        df["pickup_hour"].nunique(),
        df["pickup_hour"].min(),
        df["pickup_hour"].max(),
        zone_counts.min(),
        zone_counts.max(),
        df.duplicated(["LocationID", "pickup_hour"]).sum(),
        df.isna().sum().sum()
    ]
})

display(observation_summary)

**Interpretation:** Each modelling observation represents one taxi zone at one hourly timestamp. Taxi-demand analysis therefore uses zone-hour rows, weather summaries use one record per unique hour, and static contextual variables are examined once per taxi zone.

## 3. Hourly Taxi Demand Distribution

In [ ]:
pickups = df["pickup_count"]

target_statistics = pd.DataFrame({
    "Statistic": [
        "Total pickups",
        "Mean",
        "Standard deviation",
        "Minimum",
        "25th percentile",
        "Median",
        "75th percentile",
        "90th percentile",
        "95th percentile",
        "99th percentile",
        "Maximum",
        "Skewness",
        "Zero-demand zone-hours",
        "Zero-demand zone-hours (%)"
    ],
    "Value": [
        pickups.sum(),
        pickups.mean(),
        pickups.std(),
        pickups.min(),
        pickups.quantile(0.25),
        pickups.quantile(0.50),
        pickups.quantile(0.75),
        pickups.quantile(0.90),
        pickups.quantile(0.95),
        pickups.quantile(0.99),
        pickups.max(),
        pickups.skew(),
        pickups.eq(0).sum(),
        pickups.eq(0).mean() * 100
    ]
})

display(target_statistics)

zero_demand_zones = (
    df.groupby(["LocationID", "borough", "zone"], as_index=False)["pickup_count"]
      .sum()
      .query("pickup_count == 0")
)

display(zero_demand_zones)

In [ ]:
upper_99 = pickups.quantile(0.99)

fig, ax = plt.subplots(figsize=(9.5, 5.6))
ax.hist(df.loc[pickups <= upper_99, "pickup_count"], bins=60)
ax.set_yscale("log")
ax.set_xlabel("Pickups per taxi-zone hour (0-99th percentile)")
ax.set_ylabel("Frequency - logarithmic scale")
ax.set_title("Distribution of Hourly Yellow Taxi Pickup Counts")
plt.tight_layout()
plt.show()

**Interpretation:** The target is strongly right-skewed, with a large concentration of zero and very low pickup counts together with a much smaller high-demand tail.

## 4. Temporal Taxi Demand Patterns

### 4.1 Demand by Hour of Day

In [ ]:
hour_summary = (
    df.groupby("hour")["pickup_count"]
      .agg(mean_pickups="mean", median_pickups="median", total_pickups="sum")
      .reset_index()
)

display(hour_summary)

hour_extremes = pd.DataFrame({
    "Measure": ["Lowest mean-demand hour", "Highest mean-demand hour"],
    "Hour": [
        int(hour_summary.loc[hour_summary["mean_pickups"].idxmin(), "hour"]),
        int(hour_summary.loc[hour_summary["mean_pickups"].idxmax(), "hour"])
    ],
    "Mean pickups": [
        hour_summary["mean_pickups"].min(),
        hour_summary["mean_pickups"].max()
    ]
})

display(hour_extremes)

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.plot(hour_summary["hour"], hour_summary["mean_pickups"], marker="o")
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("Hour of day")
ax.set_ylabel("Mean pickups per zone-hour")
ax.set_title("Mean Yellow Taxi Demand by Hour of Day")
plt.tight_layout()
plt.show()

**Interpretation:** Taxi demand follows a clear daily cycle, falling to its lowest level in the early morning and reaching its strongest average level during the evening period.

### 4.2 Demand by Day of Week

In [ ]:
day_labels = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday"
}

day_summary = (
    df.groupby("day_of_week")["pickup_count"]
      .agg(mean_pickups="mean", median_pickups="median", total_pickups="sum")
      .reindex(range(7))
      .reset_index()
)
day_summary["day"] = day_summary["day_of_week"].map(day_labels)

display(day_summary[["day_of_week", "day", "mean_pickups", "median_pickups", "total_pickups"]])

hour_day = (
    df.groupby(["day_of_week", "hour"])["pickup_count"]
      .mean()
      .unstack("hour")
      .reindex(range(7))
)

hour_day_long = hour_day.stack().rename("mean_pickups").reset_index()
highest_day_hour = hour_day_long.loc[hour_day_long["mean_pickups"].idxmax()].copy()
lowest_day_hour = hour_day_long.loc[hour_day_long["mean_pickups"].idxmin()].copy()

hour_day_extremes = pd.DataFrame({
    "Measure": ["Lowest day-hour combination", "Highest day-hour combination"],
    "Day": [day_labels[int(lowest_day_hour["day_of_week"])], day_labels[int(highest_day_hour["day_of_week"])]],
    "Hour": [int(lowest_day_hour["hour"]), int(highest_day_hour["hour"])],
    "Mean pickups": [lowest_day_hour["mean_pickups"], highest_day_hour["mean_pickups"]]
})

display(hour_day_extremes)

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 5.4))
im = ax.imshow(hour_day.values, aspect="auto")
ax.set_xticks(range(24))
ax.set_xticklabels(range(24), fontsize=8)
ax.set_yticks(range(7))
ax.set_yticklabels([day_labels[i] for i in range(7)])
ax.set_xlabel("Hour of day")
ax.set_ylabel("Day of week")
ax.set_title("Mean Taxi Demand by Hour and Day of Week")
fig.colorbar(im, ax=ax, label="Mean pickups per zone-hour")
plt.tight_layout()
plt.show()

**Interpretation:** The heatmap shows that the hourly demand cycle changes across the week, with stronger late-afternoon and evening activity on several days and different weekend behaviour.

### 4.3 Demand by Month

In [ ]:
month_labels = {1: "January", 2: "February", 3: "March"}

month_summary = (
    df.groupby("month")["pickup_count"]
      .agg(mean_pickups="mean", median_pickups="median", total_pickups="sum")
      .reset_index()
)
month_summary["month_name"] = month_summary["month"].map(month_labels)

display(month_summary[["month", "month_name", "mean_pickups", "median_pickups", "total_pickups"]])

**Interpretation:** The three months provide the model with the temporal position within Q1 2025, but they are not sufficient to represent a complete annual seasonal pattern.

## 5. Spatial Variation in Taxi Demand

### 5.1 Demand by Borough

In [ ]:
borough_summary = (
    df.groupby("borough")
      .agg(
          taxi_zones=("LocationID", "nunique"),
          total_pickups=("pickup_count", "sum"),
          mean_zone_hour_demand=("pickup_count", "mean"),
          median_zone_hour_demand=("pickup_count", "median")
      )
      .sort_values("total_pickups", ascending=False)
)
borough_summary["pickup_share_percent"] = (
    borough_summary["total_pickups"] / borough_summary["total_pickups"].sum() * 100
)

display(borough_summary.reset_index())

In [ ]:
plot_borough = borough_summary.reset_index()

fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.bar(plot_borough["borough"], plot_borough["pickup_share_percent"])
ax.set_xlabel("Borough")
ax.set_ylabel("Share of Q1 Yellow Taxi pickups (%)")
ax.set_title("Yellow Taxi Pickup Share by Borough")
plt.tight_layout()
plt.show()

**Interpretation:** Yellow Taxi demand is highly concentrated in Manhattan, showing why aggregate citywide model metrics can be dominated by a relatively small spatial part of the study area.

### 5.2 Demand by Taxi Zone

In [ ]:
zone_summary = (
    df.groupby(["LocationID", "borough", "zone"], as_index=False)["pickup_count"]
      .agg(mean_hourly_demand="mean", median_hourly_demand="median", total_pickups="sum")
)

zone_distribution = zone_summary["mean_hourly_demand"].describe(percentiles=[0.25, 0.50, 0.75, 0.90]).to_frame("mean_hourly_demand")
zone_demand_skewness = zone_summary["mean_hourly_demand"].skew()

display(zone_distribution)
display(pd.DataFrame({"Statistic": ["Mean hourly demand across zones", "Median hourly demand across zones", "Skewness"], "Value": [zone_summary["mean_hourly_demand"].mean(), zone_summary["mean_hourly_demand"].median(), zone_demand_skewness]}))

display(zone_summary.sort_values("mean_hourly_demand", ascending=False).head(10))

borough_zone_median = (
    zone_summary.groupby("borough")["mean_hourly_demand"]
                .median()
                .rename("median_zone_mean_hourly_demand")
                .sort_values(ascending=False)
                .reset_index()
)
display(borough_zone_median)

In [ ]:
borough_order = ["Bronx", "Brooklyn", "Manhattan", "Queens", "Staten Island"]
box_data = [
    np.log1p(zone_summary.loc[zone_summary["borough"] == borough, "mean_hourly_demand"].values)
    for borough in borough_order
]

fig, ax = plt.subplots(figsize=(10.0, 5.4))
ax.boxplot(box_data, tick_labels=borough_order, showfliers=True)
ax.set_xlabel("Borough")
ax.set_ylabel("log(1 + mean hourly pickups)")
ax.set_title("Variation in Mean Hourly Demand Across Taxi Zones")
plt.tight_layout()
plt.show()

**Interpretation:** Mean hourly demand varies substantially between taxi zones and is strongly right-skewed. Most zones have relatively low average demand, while a smaller number of zones have very high demand. This supports evaluating model performance separately for individual taxi zones and leads directly to examining the zone-level contextual characteristics in the next section.

## 6. Contextual Taxi-Zone Characteristics

The spatial analysis above shows substantial differences in mean hourly demand between taxi zones. The contextual variables are therefore examined next at the same taxi-zone level because their values remain fixed across the hourly observations belonging to each zone.

In [ ]:
zone_context = (
    df.sort_values(["LocationID", "pickup_hour"])
      .drop_duplicates("LocationID")
      [[
          "LocationID", "borough", "zone", "subway_proximity_km",
          "estimated_total_households", "estimated_zero_vehicle_households",
          "zero_vehicle_household_rate", "zero_vehicle_rate_moe",
          "zero_vehicle_rate_cv", "census_reliability", "census_coverage_ratio"
      ]]
      .merge(
          zone_summary[["LocationID", "mean_hourly_demand", "total_pickups"]],
          on="LocationID",
          how="left",
          validate="one_to_one"
      )
)

### 6.1 Subway Proximity

In [ ]:
subway_statistics = zone_context["subway_proximity_km"].describe(percentiles=[0.25, 0.50, 0.75, 0.90]).to_frame("subway_proximity_km")
display(subway_statistics)

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.hist(zone_context["subway_proximity_km"], bins=30)
ax.set_xlabel("Distance to nearest retained MTA station (km)")
ax.set_ylabel("Taxi zones")
ax.set_title("Distribution of Subway Proximity Across Taxi Zones")
plt.tight_layout()
plt.show()

**Interpretation:** Most taxi zones are relatively close to a retained MTA station, while a smaller group forms a long right tail of greater station distance.

### 6.2 Zero-Vehicle Household Rate

In [ ]:
zero_vehicle_statistics = (zone_context["zero_vehicle_household_rate"] * 100).describe(percentiles=[0.25, 0.50, 0.75, 0.90]).to_frame("zero_vehicle_household_rate_percent")
display(zero_vehicle_statistics)

borough_zero_vehicle = (
    zone_context.assign(zero_vehicle_household_rate_percent=zone_context["zero_vehicle_household_rate"] * 100)
                .groupby("borough")["zero_vehicle_household_rate_percent"]
                .median()
                .rename("median_zero_vehicle_household_rate_percent")
                .sort_values(ascending=False)
                .reset_index()
)
display(borough_zero_vehicle)

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.hist(zone_context["zero_vehicle_household_rate"] * 100, bins=30)
ax.set_xlabel("Estimated zero-vehicle household rate (%)")
ax.set_ylabel("Taxi zones")
ax.set_title("Distribution of Zero-Vehicle Household Rates Across Taxi Zones")
plt.tight_layout()
plt.show()

**Interpretation:** Household vehicle availability varies widely across the taxi zones, providing substantial spatial variation for the planned contextual experiment.

### 6.3 Relationship Between Contextual Features and Zone-Level Demand

The zone-level demand distribution examined in Section 5.2 is strongly right-skewed, so Spearman rank correlation is used to describe monotonic associations with the contextual variables.

In [ ]:
rho_subway, p_subway = spearmanr(zone_context["subway_proximity_km"], zone_context["mean_hourly_demand"])
rho_vehicle, p_vehicle = spearmanr(zone_context["zero_vehicle_household_rate"], zone_context["mean_hourly_demand"])
rho_context, p_context = spearmanr(zone_context["subway_proximity_km"], zone_context["zero_vehicle_household_rate"])

context_correlations = pd.DataFrame({
    "Relationship": [
        "Subway proximity vs mean hourly taxi demand",
        "Zero-vehicle household rate vs mean hourly taxi demand",
        "Subway proximity vs zero-vehicle household rate"
    ],
    "Spearman rho": [rho_subway, rho_vehicle, rho_context],
    "p-value": [p_subway, p_vehicle, p_context]
})

display(context_correlations)

display(
    zone_context.sort_values("mean_hourly_demand", ascending=False)
                [["LocationID", "borough", "zone", "mean_hourly_demand", "subway_proximity_km", "zero_vehicle_household_rate"]]
                .head(10)
)

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.4))
ax.scatter(zone_context["subway_proximity_km"], np.log1p(zone_context["mean_hourly_demand"]))
ax.set_xlabel("Subway proximity (km)")
ax.set_ylabel("log(1 + mean hourly pickups)")
ax.set_title(f"Subway Proximity and Taxi-Zone Demand (Spearman rho = {rho_subway:.3f})")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.4))
ax.scatter(zone_context["zero_vehicle_household_rate"] * 100, np.log1p(zone_context["mean_hourly_demand"]))
ax.set_xlabel("Zero-vehicle household rate (%)")
ax.set_ylabel("log(1 + mean hourly pickups)")
ax.set_title(f"Zero-Vehicle Household Rate and Taxi-Zone Demand (Spearman rho = {rho_vehicle:.3f})")
plt.tight_layout()
plt.show()

**Interpretation:** Both contextual variables show meaningful rank relationships with zone-level demand, but the scatterplots also contain clear exceptions. Their predictive contribution must therefore be tested through the controlled model experiments rather than inferred from correlation alone.

## 7. Weather Characteristics and Taxi Demand

Weather summaries are calculated from one observation per unique hourly timestamp because the same citywide weather value is repeated across all taxi zones for each hour.

### 7.1 Temperature

In [ ]:
weather = (
    df[["pickup_hour", "temperature_f", "precipitation_in"]]
      .drop_duplicates("pickup_hour")
      .sort_values("pickup_hour")
)

temperature_statistics = weather["temperature_f"].describe().to_frame("temperature_f")
display(temperature_statistics)

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.hist(weather["temperature_f"], bins=35)
ax.set_xlabel("Temperature (°F)")
ax.set_ylabel("Hourly observations")
ax.set_title("Distribution of Hourly Temperature")
plt.tight_layout()
plt.show()

**Interpretation:** Temperature provides continuous hourly variation across the Q1 study period, although the observed range mainly represents winter and early-spring conditions.

### 7.2 Precipitation

In [ ]:
precipitation_summary = pd.DataFrame({
    "Measure": [
        "Unique hourly observations",
        "No-precipitation hours",
        "No-precipitation hours (%)",
        "Positive-precipitation hours",
        "Positive-precipitation hours (%)",
        "Mean precipitation",
        "Median precipitation",
        "Median positive precipitation",
        "Maximum precipitation"
    ],
    "Value": [
        len(weather),
        weather["precipitation_in"].eq(0).sum(),
        weather["precipitation_in"].eq(0).mean() * 100,
        weather["precipitation_in"].gt(0).sum(),
        weather["precipitation_in"].gt(0).mean() * 100,
        weather["precipitation_in"].mean(),
        weather["precipitation_in"].median(),
        weather.loc[weather["precipitation_in"] > 0, "precipitation_in"].median(),
        weather["precipitation_in"].max()
    ]
})

display(precipitation_summary)

In [ ]:
precipitation_occurrence = pd.Series({
    "No precipitation": weather["precipitation_in"].eq(0).sum(),
    "Positive precipitation": weather["precipitation_in"].gt(0).sum()
})

fig, ax = plt.subplots(figsize=(8.5, 5.0))
ax.bar(precipitation_occurrence.index, precipitation_occurrence.values)
ax.set_ylabel("Hourly observations")
ax.set_title("Occurrence of Hourly Precipitation During Q1 2025")
plt.tight_layout()
plt.show()

In [ ]:
city_hourly_demand = (
    df.groupby("pickup_hour", as_index=False)["pickup_count"]
      .sum()
      .merge(weather, on="pickup_hour", how="left", validate="one_to_one")
)

weather_correlations = pd.DataFrame({
    "Relationship": [
        "Temperature vs citywide hourly pickups",
        "Precipitation vs citywide hourly pickups"
    ],
    "Pearson correlation": [
        pearsonr(city_hourly_demand["temperature_f"], city_hourly_demand["pickup_count"])[0],
        pearsonr(city_hourly_demand["precipitation_in"], city_hourly_demand["pickup_count"])[0]
    ]
})

display(weather_correlations)

**Interpretation:** Positive precipitation is uncommon during the study period. The simple weather correlations are weak, but they do not rule out nonlinear relationships that can be learned by tree-based models.

## 8. Census Feature Reliability

In [ ]:
reliability_order = ["high", "moderate", "low"]

reliability_summary = (
    zone_context["census_reliability"]
      .value_counts()
      .reindex(reliability_order, fill_value=0)
      .rename_axis("reliability")
      .reset_index(name="taxi_zones")
)
reliability_summary["percentage"] = reliability_summary["taxi_zones"] / len(zone_context) * 100

display(reliability_summary)

coverage_statistics = zone_context["census_coverage_ratio"].describe().to_frame("census_coverage_ratio")
display(coverage_statistics)

low_reliability_zones = (
    zone_context.loc[zone_context["census_reliability"].eq("low"), [
        "LocationID", "borough", "zone", "estimated_total_households",
        "zero_vehicle_household_rate", "zero_vehicle_rate_moe",
        "zero_vehicle_rate_cv", "census_coverage_ratio"
    ]]
    .sort_values("zero_vehicle_rate_cv", ascending=False)
)
display(low_reliability_zones)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.0))
ax.bar(reliability_summary["reliability"].str.title(), reliability_summary["taxi_zones"])
ax.set_xlabel("Census reliability category")
ax.set_ylabel("Taxi zones")
ax.set_title("Reliability of Zero-Vehicle Household Estimates")
plt.tight_layout()
plt.show()

**Interpretation:** Most taxi-zone estimates fall in the high-reliability category, while the low-reliability cases are concentrated in a small group that can be retained for quality interpretation without adding the uncertainty measures as predictors.

## 9. Repeated Taxi-Zone Structure

In [ ]:
repeated_zone_check = (
    df.groupby("LocationID")
      .agg(
          rows=("pickup_hour", "size"),
          unique_hours=("pickup_hour", "nunique"),
          unique_subway_values=("subway_proximity_km", "nunique"),
          unique_zero_vehicle_rates=("zero_vehicle_household_rate", "nunique")
      )
)

repeated_structure_summary = pd.DataFrame({
    "Check": [
        "Rows per LocationID - minimum",
        "Rows per LocationID - maximum",
        "Unique hours per LocationID - minimum",
        "Unique hours per LocationID - maximum",
        "Unique subway values within each zone - maximum",
        "Unique zero-vehicle rates within each zone - maximum",
        "Distinct subway values across zones",
        "Distinct zero-vehicle rates across zones"
    ],
    "Result": [
        repeated_zone_check["rows"].min(),
        repeated_zone_check["rows"].max(),
        repeated_zone_check["unique_hours"].min(),
        repeated_zone_check["unique_hours"].max(),
        repeated_zone_check["unique_subway_values"].max(),
        repeated_zone_check["unique_zero_vehicle_rates"].max(),
        zone_context["subway_proximity_km"].nunique(),
        zone_context["zero_vehicle_household_rate"].nunique()
    ]
})

display(repeated_structure_summary)

**Interpretation:** Every taxi zone is observed repeatedly through time, while the two contextual variables remain fixed within a zone. `LocationID` is therefore retained for grouping and zone-level evaluation rather than included in the primary predictor matrix.

## 10. Definition of Low-Demand Outer-Borough Zones

In [ ]:
outer_boroughs = ["Bronx", "Brooklyn", "Queens", "Staten Island"]

zone_demand = (
    df.groupby(["LocationID", "borough", "zone"], as_index=False)["pickup_count"]
      .mean()
      .rename(columns={"pickup_count": "mean_hourly_demand"})
)

outer_zone_demand = zone_demand.loc[
    zone_demand["borough"].isin(outer_boroughs)
].copy()

q1 = outer_zone_demand["mean_hourly_demand"].quantile(0.25)
median = outer_zone_demand["mean_hourly_demand"].median()
q3 = outer_zone_demand["mean_hourly_demand"].quantile(0.75)

outer_zone_demand["demand_group"] = np.where(
    outer_zone_demand["mean_hourly_demand"] <= q1,
    "Low demand",
    "Other"
)

low_demand_outer = outer_zone_demand.loc[
    outer_zone_demand["demand_group"].eq("Low demand")
].copy()

low_demand_statistics = pd.DataFrame({
    "Measure": [
        "Outer-borough taxi zones",
        "Minimum mean hourly demand",
        "Mean hourly demand",
        "First quartile (Q1)",
        "Median",
        "Third quartile (Q3)",
        "Maximum mean hourly demand",
        "Skewness",
        "Low-demand outer-borough zones"
    ],
    "Value": [
        len(outer_zone_demand),
        outer_zone_demand["mean_hourly_demand"].min(),
        outer_zone_demand["mean_hourly_demand"].mean(),
        q1,
        median,
        q3,
        outer_zone_demand["mean_hourly_demand"].max(),
        outer_zone_demand["mean_hourly_demand"].skew(),
        len(low_demand_outer)
    ]
})

display(low_demand_statistics)

low_demand_by_borough = (
    low_demand_outer.groupby("borough")
                    .size()
                    .reindex(outer_boroughs, fill_value=0)
                    .rename("low_demand_zones")
                    .reset_index()
)

display(low_demand_by_borough)
display(low_demand_outer.sort_values(["borough", "mean_hourly_demand", "LocationID"]))

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.4))

ax.boxplot(
    outer_zone_demand["mean_hourly_demand"],
    vert=False,
    showfliers=False,
    widths=0.42
)

ax.axvline(
    q1,
    linestyle="--",
    linewidth=1.5,
    label=f"Q1 low-demand threshold = {q1:.4f}"
)

ax.set_xlabel("Mean hourly Yellow Taxi pickups per outer-borough zone")
ax.set_yticks([])
ax.set_title("Outer-Borough Mean Hourly Taxi Demand")

upper_visible = q3 + 1.75 * (q3 - q1)
ax.set_xlim(left=0, right=max(4.0, upper_visible))

ax.text(q1, 1.20, f"Q1 = {q1:.4f}", ha="center", va="bottom")
ax.text(median, 0.76, f"Median = {median:.4f}", ha="center", va="top")
ax.text(q3, 1.20, f"Q3 = {q3:.4f}", ha="center", va="bottom")

ax.legend(loc="upper right")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

**Interpretation:** The outer-borough demand distribution is strongly right-skewed, so the low-demand group is defined using the first quartile rather than an arbitrary pickup-count cutoff. The box plot shows Q1, the median, and Q3 directly. Zones at or below Q1 form the lowest 25% of the outer-borough distribution. Outlier points are hidden only to keep the quartile box readable; they remain included in all calculations. The resulting low-demand label is used later for grouped model evaluation and is not used as a model predictor.

## 11. Prepare Baseline and Enriched Modelling Datasets


In [ ]:
# Columns retained to identify each prediction later during zone-level evaluation
identifier_columns = [
    "LocationID",
    "borough",
    "zone",
    "pickup_hour"
]

target_column = "pickup_count"

# Baseline predictors: temporal + weather features
baseline_features = [
    "hour",
    "day_of_week",
    "month",
    "temperature_f",
    "precipitation_in"
]

# Contextual features added to the enriched dataset
contextual_features = [
    "subway_proximity_km",
    "zero_vehicle_household_rate"
]

# 1. Baseline modelling dataset
baseline_model_dataset = df[
    identifier_columns + [target_column] + baseline_features
].copy()

# 2. Enriched modelling dataset
enriched_model_dataset = df[
    identifier_columns + [target_column] + baseline_features + contextual_features
].copy()

# Confirm the two datasets created for the Model Building phase
dataset_summary = pd.DataFrame({
    "Dataset": ["Baseline", "Enriched"],
    "Rows": [baseline_model_dataset.shape[0], enriched_model_dataset.shape[0]],
    "Columns": [baseline_model_dataset.shape[1], enriched_model_dataset.shape[1]],
    "Taxi Zones": [
        baseline_model_dataset["LocationID"].nunique(),
        enriched_model_dataset["LocationID"].nunique()
    ],
    "Model Predictors": [len(baseline_features), len(baseline_features + contextual_features)]
})

display(dataset_summary)

display(baseline_model_dataset.head())
display(enriched_model_dataset.head())
